# IY029: Pairwise Embedding Space Visualisation

Visualises the latent embedding space of all 6 pairwise same/different methods on the
IY029 v2 test sets (n = 2 000 pairs per condition × fold).

| Method | Embedding | Dim |
|---|---|---|
| Raw SVM | StandardScaler(flatten(pair)) | T (variable) |
| catch22 + SVM | StandardScaler([catch22(x1) ∣ catch22(x2)]) | 44 |
| SimCLR + SVM | backbone.encode([z1 ∣ z2]) | 2 × d_model |
| MLP | Penultimate layer activations (ReLU → 64-dim) | 64 |
| Transformer | model.encode() (mean-pooled d_model) | 16 |
| LSTM | Context vector before fc_layers (bidirectional) | 128 |

**Point** = one test-set pair (two concatenated trajectories).  
**Colour** = `same` (y = 1, blue) vs `different` (y = 0, orange/red).  
**Layout** = 4 conditions (rows) × 2 folds (columns) per method.

Dimensionality reduction:
- **PCA**: linear projection to 2D (applied directly)
- **t-SNE**: PCA-50 pre-reduction (for high-dim embeddings) then t-SNE

Summary: silhouette score heatmap across all methods × conditions × folds.

In [1]:
import sys
import re
import json
import time
import warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import pycatch22
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

# ── Paths ─────────────────────────────────────────────────────────────────────
SCRIPT_DIR = Path('.')
REPO_ROOT  = SCRIPT_DIR.resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / 'src'))

from dataloaders import load_loader_from_disk
from models.ssl_transformer import SSL_Transformer
from models.lstm import LSTMClassifier
from models.transformer import TransformerClassifier

IY029_DATA = SCRIPT_DIR / 'data'       # EXP-26-IY029/data/{2_fold,10_fold}/<cond>/
CKPT_DIR   = SCRIPT_DIR / 'checkpoints'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'REPO_ROOT  : {REPO_ROOT}')
print(f'IY029_DATA : {IY029_DATA}  (exists: {IY029_DATA.exists()})')
print(f'CKPT_DIR   : {CKPT_DIR}  (exists: {CKPT_DIR.exists()})')
print(f'Device     : {DEVICE}')

REPO_ROOT  : /home/ianyang/stochastic_simulations
IY029_DATA : data  (exists: True)
CKPT_DIR   : checkpoints  (exists: False)
Device     : cuda


In [2]:
from utils.embeddings import make_checkpoint_arch_label
from utils.embeddings import parse_arch_from_name as parse_arch
# ── Dataset & display config ──────────────────────────────────────────────────
CONDITIONS   = ['baseline', 'mu', 'cv', 't_ac']
FOLDS        = ['2_fold', '10_fold']
COND_NAMES   = ['Baseline', 'Mu', 'CV', 'T_ac']        # display names (row labels)
FOLD_DISPLAY = ['2-fold', '10-fold']                   # display names (col labels)
N_JOBS_C22   = -1                                      # joblib workers for catch22

# ── Plotting style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':     'sans-serif',
    'axes.labelsize':  10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 10,
    'axes.titlesize':  11,
})
palette   = sns.color_palette('colorblind')
COL_SAME  = palette[0]   # blue  — same (y=1)
COL_DIFF  = palette[3]   # vermilion — different (y=0)

# ── SimCLR model registry ─────────────────────────────────────────────────────
NORM_OVERRIDES = {
    'IY022_simCLR_b64_lr0.01_L2_H4_D16_20260303_170229_model': 'global',
    'IY022_simCLR_b64_lr0.01_L2_H4_D16_20260308_125632_model': 'global',
    'IY022_simCLR_b64_lr0.01_L2_H4_D16_20260308_132219_model': 'joint',
    'IY023_simCLR_b64_lr0.01_L2_H4_D16_20260308_125550_model': 'global',
    'IY023_simCLR_b64_lr0.01_L2_H4_D16_20260308_165118_model': 'joint',
}
CKPT_DIRS = ['EXP-26-IY017', 'EXP-26-IY022', 'EXP-26-IY023', 'EXP-26-IY024']

MODEL_REGISTRY = [
    (p, make_checkpoint_arch_label(p, NORM_OVERRIDES))
    for d in CKPT_DIRS
    for p in sorted((REPO_ROOT / 'experiments' / d).glob('*.pth'))
]
print(f'SimCLR checkpoints found: {len(MODEL_REGISTRY)}')

SimCLR checkpoints found: 34


## Helper definitions

In [3]:
from features.catch22 import catch22_features

# ── Data loading ──────────────────────────────────────────────────────────────

def _load_pt(pt_path: Path):
    """Load a .pt static loader → (X_np, y_np) numpy arrays."""
    loader = load_loader_from_disk(pt_path, batch_size=2048)
    Xs, ys = [], []
    for X, y in loader:
        Xs.append(X.numpy()); ys.append(y.numpy().ravel())
    return np.concatenate(Xs), np.concatenate(ys).astype(int)


def load_split(cond: str, fold: str, split: str):
    """
    Load one (cond, fold, split) triple from the IY029 v2 .pt files.
    split ∈ {'train', 'val', 'test'}.
    """
    pt = IY029_DATA / fold / cond / f'IY029_static_{split}.pt'
    if not pt.exists():
        raise FileNotFoundError(f'File not found: {pt}\n'
                                f'Run IY029_regen_pt_datasets.sh on HPC first.')
    return _load_pt(pt)


# ── MLP architecture (must match IY029_mlp_pairwise_v2.py) ────────────────────

class PairwiseMLP(nn.Module):
    """Two-layer MLP for pairwise same/different; expose encode() for penultimate layer."""
    def __init__(self, input_dim, hidden_dims=(512, 256, 64), dropout=0.3):
        super().__init__()
        layers, in_dim = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

    def encode(self, x):
        """64-dim penultimate activations (Dropout is identity in eval mode)."""
        return self.net[:-1](x)


# ── catch22 feature helpers ────────────────────────────────────────────────────




In [4]:
# ── Per-method embedding extractors ──────────────────────────────────────────
# Each function takes (X_test_np, *optional_model) and returns (N_test, D) float32.

def embed_svm(X_train_np: np.ndarray, X_test_np: np.ndarray) -> np.ndarray:
    """Flatten pairs and z-score using a StandardScaler fit on the training set."""
    X_tr = X_train_np.reshape(len(X_train_np), -1)
    X_te = X_test_np.reshape(len(X_test_np),   -1)
    scaler = StandardScaler()
    scaler.fit(X_tr)
    return scaler.transform(X_te).astype(np.float32)   # (N, T)


def embed_catch22(X_train_np: np.ndarray, X_test_np: np.ndarray) -> np.ndarray:
    """Compute [catch22(x1)|catch22(x2)] and z-score using scaler fit on train."""
    print('    catch22 train...', end='', flush=True); t0 = time.time()
    f_tr = catch22_features(X_train_np)
    print(f' {time.time()-t0:.0f}s | test...', end='', flush=True); t0 = time.time()
    f_te = catch22_features(X_test_np)
    print(f' {time.time()-t0:.0f}s', flush=True)
    scaler = StandardScaler()
    scaler.fit(f_tr)
    return scaler.transform(f_te).astype(np.float32)   # (N, 44)


def embed_simclr(X_test_np: np.ndarray, model: SSL_Transformer,
                 batch_size: int = 128) -> np.ndarray:
    """Encode pairs as [z1|z2] using best SimCLR backbone. Returns (N, 2*d_model)."""
    model.eval()
    half = X_test_np.shape[1] // 2
    x1 = X_test_np[:, :half, :]
    x2 = X_test_np[:, half:, :]
    z1_list, z2_list = [], []
    with torch.no_grad():
        for i in range(0, len(X_test_np), batch_size):
            b1 = torch.tensor(x1[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            b2 = torch.tensor(x2[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            z1_list.append(model.backbone.encode(b1).cpu().numpy())
            z2_list.append(model.backbone.encode(b2).cpu().numpy())
    z1 = np.concatenate(z1_list); z2 = np.concatenate(z2_list)
    return np.concatenate([z1, z2], axis=1).astype(np.float32)   # (N, 2*d_model)


def embed_mlp(X_test_np: np.ndarray, model: PairwiseMLP,
              batch_size: int = 512) -> np.ndarray:
    """Extract 64-dim penultimate layer via model.encode()."""
    model.eval()
    X_flat = X_test_np.reshape(len(X_test_np), -1)
    X_t    = torch.tensor(X_flat, dtype=torch.float32)
    parts  = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            parts.append(model.encode(X_t[i:i+batch_size].to(DEVICE)).cpu().numpy())
    return np.concatenate(parts).astype(np.float32)   # (N, 64)


def embed_transformer(X_test_np: np.ndarray, model: TransformerClassifier,
                      batch_size: int = 32) -> np.ndarray:
    """Extract d_model-dim embedding via model.encode() (mean-pooled Transformer output)."""
    model.eval()
    X_t   = torch.tensor(X_test_np, dtype=torch.float32)
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            parts.append(model.encode(X_t[i:i+batch_size].to(DEVICE)).cpu().numpy())
    return np.concatenate(parts).astype(np.float32)   # (N, d_model)


def embed_lstm(X_test_np: np.ndarray, model: LSTMClassifier,
               batch_size: int = 64) -> np.ndarray:
    """
    Extract 128-dim pooled LSTM context vector via a forward pre-hook on fc_layers.
    The hook captures the input to fc_layers before the classification head.
    """
    model.eval()
    X_t      = torch.tensor(X_test_np, dtype=torch.float32)
    contexts = []

    def _hook(module, inputs):
        # inputs[0] has shape (B, lstm_output_size) = (B, 128) for bidirectional LSTM
        contexts.append(inputs[0].detach().cpu().numpy())

    handle = model.fc_layers.register_forward_pre_hook(_hook)
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            model(X_t[i:i+batch_size].to(DEVICE))
    handle.remove()
    return np.concatenate(contexts).astype(np.float32)   # (N, 128)

## Load data & models

In [5]:
# ── Load all 8 test sets and train sets (train needed for SVM/catch22 scalers) ──
print('Loading test and train sets...')
test_data  = {}   # {(cond, fold): (X_np, y_np)}
train_data = {}   # {(cond, fold): (X_np, y_np)}  — for scaler fitting

for cond in CONDITIONS:
    for fold in FOLDS:
        print(f'  {cond} / {fold}...', end=' ', flush=True)
        test_data[(cond, fold)]  = load_split(cond, fold, 'test')
        train_data[(cond, fold)] = load_split(cond, fold, 'train')
        X_te, y_te = test_data[(cond, fold)]
        print(f'test={X_te.shape}  same={y_te.sum()}  diff={(y_te==0).sum()}')

print(f'\nLoaded {len(test_data)} test sets.')

Loading test and train sets...
  baseline / 2_fold... 📂 Loading static data from data/2_fold/baseline/IY029_static_test.pt...
📂 Loading static data from data/2_fold/baseline/IY029_static_train.pt...
test=(2000, 3623, 1)  same=992  diff=1008
  baseline / 10_fold... 📂 Loading static data from data/10_fold/baseline/IY029_static_test.pt...
📂 Loading static data from data/10_fold/baseline/IY029_static_train.pt...
test=(2000, 3623, 1)  same=969  diff=1031
  mu / 2_fold... 📂 Loading static data from data/2_fold/mu/IY029_static_test.pt...
📂 Loading static data from data/2_fold/mu/IY029_static_train.pt...
test=(2000, 5021, 1)  same=1009  diff=991
  mu / 10_fold... 📂 Loading static data from data/10_fold/mu/IY029_static_test.pt...
📂 Loading static data from data/10_fold/mu/IY029_static_train.pt...
test=(2000, 5021, 1)  same=995  diff=1005
  cv / 2_fold... 📂 Loading static data from data/2_fold/cv/IY029_static_test.pt...
📂 Loading static data from data/2_fold/cv/IY029_static_train.pt...
test=(200

In [6]:
# ── Load SimCLR best model ────────────────────────────────────────────────────
# Prefer the v2 results JSON; fall back to the original.
def _load_simclr_best():
    for json_path in [
        SCRIPT_DIR / 'IY029_simclr_svm_pairwise_v2_results.json',
        SCRIPT_DIR / 'IY029_simclr_svm_pairwise_results.json',
    ]:
        if json_path.exists():
            best_label = json.load(open(json_path))['best_label']
            print(f'SimCLR best label (from {json_path.name}): {best_label}')
            # Find the matching checkpoint path in the registry
            matches = [(p, lbl) for p, lbl in MODEL_REGISTRY if lbl == best_label]
            if matches:
                ckpt_path, _ = matches[0]
                arch  = parse_arch(ckpt_path.stem)
                model = SSL_Transformer(**arch).to(DEVICE)
                state = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
                model.load_state_dict(state)
                model.eval()
                print(f'  Loaded: {ckpt_path.name}')
                return model, best_label
    # No JSON found — pick best by mean accuracy across the registry
    print('No SimCLR results JSON found; using first checkpoint in registry.')
    ckpt_path, lbl = MODEL_REGISTRY[0]
    arch  = parse_arch(ckpt_path.stem)
    model = SSL_Transformer(**arch).to(DEVICE)
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    return model, lbl

simclr_model, simclr_label = _load_simclr_best()

# ── Load MLP, LSTM, Transformer checkpoints (skip gracefully if missing) ──────
loaded_models = {}   # {('mlp'/'lstm'/'transformer', cond, fold): model}

for cond in CONDITIONS:
    for fold in FOLDS:
        key_sfx = f'{cond}_{fold}'

        # MLP
        ckpt = CKPT_DIR / f'IY029_mlp_v2_{key_sfx}.pth'
        if ckpt.exists():
            data      = torch.load(ckpt, map_location='cpu', weights_only=False)
            mlp       = PairwiseMLP(data['input_dim']).eval().to(DEVICE)
            mlp.load_state_dict(data['state_dict'])
            loaded_models[('mlp', cond, fold)] = mlp

        # Transformer
        ckpt = CKPT_DIR / f'IY029_transformer_v2_{key_sfx}.pth'
        if ckpt.exists():
            data  = torch.load(ckpt, map_location='cpu', weights_only=False)
            tfm   = TransformerClassifier(
                        input_size=1, d_model=data['d_model'], nhead=data['nhead'],
                        num_layers=data['num_layers'], num_classes=2,
                        dropout=data['dropout'],
                    ).eval().to(DEVICE)
            tfm.load_state_dict(data['state_dict'])
            loaded_models[('transformer', cond, fold)] = tfm

        # LSTM
        ckpt = CKPT_DIR / f'IY029_lstm_v2_{key_sfx}.pth'
        if ckpt.exists():
            state = torch.load(ckpt, map_location='cpu', weights_only=False)
            lstm  = LSTMClassifier(
                        input_size=1, hidden_size=64, num_layers=2, output_size=2,
                        dropout_rate=0.3, learning_rate=1e-3,
                        use_conv1d=False, use_attention=False,
                        bidirectional=True, verbose=False,
                    ).eval().to(DEVICE)
            lstm.load_state_dict(state)
            loaded_models[('lstm', cond, fold)] = lstm

# Summary
for method in ('mlp', 'transformer', 'lstm'):
    n_loaded = sum(1 for k in loaded_models if k[0] == method)
    status   = '✅' if n_loaded == 8 else f'⚠️  only {n_loaded}/8'
    print(f'{method.upper():<14}: {status} checkpoints loaded')

SimCLR best label (from IY029_simclr_svm_pairwise_results.json): IY017-b64_lr0.01_L2_H4_D16-instance-163742
  Loaded: IY017_simCLR_b64_lr0.01_L2_H4_D16_20260215_163742_model.pth
MLP           : ⚠️  only 0/8 checkpoints loaded
TRANSFORMER   : ⚠️  only 0/8 checkpoints loaded
LSTM          : ⚠️  only 0/8 checkpoints loaded


## Compute embeddings

In [7]:
# embeddings[method_key][(cond, fold)] = np.ndarray (N_test, D)
# labels[(cond, fold)]                 = np.ndarray (N_test,) int
embeddings = {m: {} for m in ['svm', 'catch22', 'simclr', 'mlp', 'lstm', 'transformer']}
labels     = {}

for cond in CONDITIONS:
    for fold in FOLDS:
        key         = (cond, fold)
        X_te, y_te  = test_data[key]
        X_tr, _     = train_data[key]
        labels[key] = y_te

        print(f'\n── {cond} / {fold}  (test n={len(X_te)}) ──')

        # Raw SVM (deterministic — no model checkpoint needed)
        print('  SVM...', end=' ', flush=True); t0 = time.time()
        embeddings['svm'][key] = embed_svm(X_tr, X_te)
        print(f'{time.time()-t0:.1f}s  shape={embeddings["svm"][key].shape}')

        # catch22 (deterministic — no model checkpoint needed)
        print('  catch22...')
        embeddings['catch22'][key] = embed_catch22(X_tr, X_te)
        print(f'    shape={embeddings["catch22"][key].shape}')

        # SimCLR (always available — pre-trained checkpoints exist)
        print('  SimCLR...', end=' ', flush=True); t0 = time.time()
        embeddings['simclr'][key] = embed_simclr(X_te, simclr_model)
        print(f'{time.time()-t0:.1f}s  shape={embeddings["simclr"][key].shape}')

        # MLP (conditional on checkpoint)
        if ('mlp', cond, fold) in loaded_models:
            print('  MLP...', end=' ', flush=True); t0 = time.time()
            embeddings['mlp'][key] = embed_mlp(X_te, loaded_models[('mlp', cond, fold)])
            print(f'{time.time()-t0:.1f}s  shape={embeddings["mlp"][key].shape}')
        else:
            print('  MLP: ⚠️  checkpoint not found — skipping')

        # Transformer (conditional on checkpoint)
        if ('transformer', cond, fold) in loaded_models:
            print('  Transformer...', end=' ', flush=True); t0 = time.time()
            embeddings['transformer'][key] = embed_transformer(
                X_te, loaded_models[('transformer', cond, fold)]
            )
            print(f'{time.time()-t0:.1f}s  shape={embeddings["transformer"][key].shape}')
        else:
            print('  Transformer: ⚠️  checkpoint not found — skipping')

        # LSTM (conditional on checkpoint)
        if ('lstm', cond, fold) in loaded_models:
            print('  LSTM...', end=' ', flush=True); t0 = time.time()
            embeddings['lstm'][key] = embed_lstm(
                X_te, loaded_models[('lstm', cond, fold)]
            )
            print(f'{time.time()-t0:.1f}s  shape={embeddings["lstm"][key].shape}')
        else:
            print('  LSTM: ⚠️  checkpoint not found — skipping')

print('\n✅ All embeddings computed.')


── baseline / 2_fold  (test n=2000) ──
  SVM... 0.7s  shape=(2000, 3623)
  catch22...
    catch22 train... 30s | test... 6s
    shape=(2000, 44)
  SimCLR... 

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.26 GiB. GPU 0 has a total capacity of 7.60 GiB of which 1.07 GiB is free. Including non-PyTorch memory, this process has 6.50 GiB memory in use. Of the allocated memory 95.32 MiB is allocated by PyTorch, and 6.26 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Visualisation

PCA and t-SNE plots — one figure per method, 4 conditions × 2 folds.  
Silhouette score (same/different separation) annotated on each subplot.

In [ ]:
def reduce_2d(emb: np.ndarray, method: str = 'PCA',
              pca_pre: int = 50, tsne_perp: int = 30,
              random_state: int = 42) -> np.ndarray:
    """
    Reduce embedding matrix to 2D.
    For t-SNE: first apply PCA to min(pca_pre, D) dims to stabilise computation
    on high-dimensional inputs (e.g., raw SVM with T > 3000 features).
    """
    D = emb.shape[1]
    if method == 'PCA':
        return PCA(n_components=2, random_state=random_state).fit_transform(emb)
    else:  # t-SNE
        if D > pca_pre:
            emb = PCA(n_components=pca_pre, random_state=random_state).fit_transform(emb)
        return TSNE(n_components=2, perplexity=tsne_perp,
                    random_state=random_state, n_jobs=-1).fit_transform(emb)


def plot_embedding_grid(method_key: str, method_title: str,
                        reduction: str = 'PCA', fname: str | None = None):
    """
    4 × 2 figure (conditions × folds) for one method and one reduction.
    Skips conditions where the embedding was not computed.
    """
    emb_dict = embeddings[method_key]
    if not emb_dict:
        print(f'⚠️  No embeddings for {method_key} — skipping plot.')
        return

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    legend_handles = [
        mpatches.Patch(color=COL_SAME, label='Same (y=1)'),
        mpatches.Patch(color=COL_DIFF, label='Different (y=0)'),
    ]

    for row, cond in enumerate(CONDITIONS):
        for col, fold in enumerate(FOLDS):
            ax  = axes[row, col]
            key = (cond, fold)

            if key not in emb_dict:
                ax.text(0.5, 0.5, 'No checkpoint', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='grey')
                ax.set_title(f'{COND_NAMES[row]}  {FOLD_DISPLAY[col]}', fontsize=11)
                ax.set_xticks([]); ax.set_yticks([])
                continue

            emb = emb_dict[key]
            y   = labels[key]

            # 2-D projection
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                emb2d = reduce_2d(emb, method=reduction)

            # Silhouette score (same/different separation in ORIGINAL embedding space)
            if emb.shape[1] >= 2 and len(np.unique(y)) == 2 and emb.shape[0] > 2:
                sil = silhouette_score(emb, y, sample_size=min(2000, len(y)),
                                       random_state=42)
                sil_str = f'sil={sil:+.3f}'
            else:
                sil_str = ''

            # Scatter — plot "different" first so "same" is visible on top
            mask_diff = y == 0; mask_same = y == 1
            ax.scatter(emb2d[mask_diff, 0], emb2d[mask_diff, 1],
                       c=[COL_DIFF], s=5, alpha=0.35, linewidths=0, rasterized=True)
            ax.scatter(emb2d[mask_same, 0], emb2d[mask_same, 1],
                       c=[COL_SAME], s=5, alpha=0.35, linewidths=0, rasterized=True)

            ax.set_title(f'{COND_NAMES[row]}  {FOLD_DISPLAY[col]}'
                         + (f'\n{sil_str}' if sil_str else ''), fontsize=11)
            ax.set_xlabel(f'{reduction}1', fontsize=9)
            ax.set_ylabel(f'{reduction}2', fontsize=9)
            ax.tick_params(labelsize=8)

    fig.legend(handles=legend_handles, loc='upper right',
               bbox_to_anchor=(0.98, 0.995), fontsize=10, framealpha=0.9)
    fig.suptitle(f'{method_title} — {reduction} of test-set pairwise embeddings',
                 fontsize=14, weight='bold', y=1.002)
    plt.tight_layout()

    if fname:
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved {fname}')
    plt.show()
    plt.close(fig)

### PCA projections

In [ ]:
METHOD_DISPLAY = [
    ('svm',         'Raw SVM'),
    ('catch22',     'catch22 + SVM'),
    ('simclr',      'SimCLR + SVM'),
    ('mlp',         'MLP'),
    ('transformer', 'Transformer'),
    ('lstm',        'LSTM'),
]

for key, title in METHOD_DISPLAY:
    print(f'\n▶ PCA — {title}')
    fname = SCRIPT_DIR / f'IY029_embedding_{key}_pca.png'
    plot_embedding_grid(key, title, reduction='PCA', fname=str(fname))

### t-SNE projections

t-SNE is preceded by PCA-50 pre-reduction for embeddings with > 50 dimensions (raw SVM, LSTM).  
Each run may take several minutes per method.

In [ ]:
for key, title in METHOD_DISPLAY:
    print(f'\n▶ t-SNE — {title}')
    fname = SCRIPT_DIR / f'IY029_embedding_{key}_tsne.png'
    plot_embedding_grid(key, title, reduction='tSNE', fname=str(fname))

### Silhouette score summary heatmap

Silhouette score measures same/different cluster separation in the **original** embedding space  
(range −1 to +1; higher = better separated).  
Rows = methods, columns = condition × fold.

In [ ]:
# ── Collect silhouette scores ─────────────────────────────────────────────────
# Columns: one per (cond, fold) combination
col_labels = [f'{COND_NAMES[i]}\n{FOLD_DISPLAY[j]}'
              for i in range(len(CONDITIONS))
              for j in range(len(FOLDS))]
row_labels = [title for _, title in METHOD_DISPLAY]

sil_matrix = np.full((len(METHOD_DISPLAY), len(col_labels)), np.nan)

for r, (method_key, _) in enumerate(METHOD_DISPLAY):
    emb_dict = embeddings[method_key]
    c_idx    = 0
    for cond in CONDITIONS:
        for fold in FOLDS:
            key = (cond, fold)
            if key in emb_dict:
                emb = emb_dict[key]
                y   = labels[key]
                if emb.shape[1] >= 2 and len(np.unique(y)) == 2 and emb.shape[0] > 2:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        sil_matrix[r, c_idx] = silhouette_score(
                            emb, y, sample_size=min(2000, len(y)), random_state=42
                        )
            c_idx += 1

# ── Plot heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

sil_df  = pd.DataFrame(sil_matrix, index=row_labels, columns=col_labels)
mask    = np.isnan(sil_matrix)   # grey out missing checkpoints

sns.heatmap(
    sil_df,
    ax=ax,
    mask=mask,
    cmap='RdYlGn',
    vmin=-0.1, vmax=0.5,
    annot=True, fmt='.3f', annot_kws={'fontsize': 9},
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Silhouette score', 'shrink': 0.8},
)

# Overlay grey hatching for NaN cells (missing checkpoints)
for r in range(sil_matrix.shape[0]):
    for c in range(sil_matrix.shape[1]):
        if mask[r, c]:
            ax.add_patch(plt.Rectangle((c, r), 1, 1,
                                       fill=True, color='lightgrey',
                                       lw=0, zorder=2))
            ax.text(c + 0.5, r + 0.5, 'N/A', ha='center', va='center',
                    fontsize=8, color='grey', zorder=3)

ax.set_title('Same / Different embedding separation — Silhouette scores',
             fontsize=14, weight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=0,  labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=10)

plt.tight_layout()
fname = SCRIPT_DIR / 'IY029_embedding_silhouette_heatmap.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved {fname}')
plt.show()
plt.close(fig)